# 🇮🇳 Unemployment in India — Data Preparation & Cleaning Pipeline

**Project Title:** Unemployment in India Analysis  
**Dataset Source:** [Kaggle: Unemployment in India](https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india)  
**Team Information:**
- **Maharudra:** Raw dataset preparation, Data cleaning, Data preprocessing, Cleaned dataset generation
- **Bhavya:** Exploratory data analysis, Visualization, Findings, Final analysis/report  

---

## 1. Project Objective
The objective of this module is to engineer a robust, reproducible data preparation and cleaning pipeline for the Unemployment in India dataset. The pipeline ingests the raw survey records (`raw_data.csv`), audits schema anomalies and delimiter artifacts, handles missing values and duplicates, applies categorical normalization and ISO datetime transformations, standardizes headers to `snake_case`, and outputs the production-ready dataset (`cleaned_data.csv`).


---
## 2. Import Libraries
We import core scientific and data manipulation libraries (`pandas`, `numpy`, `pathlib`).


In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np

# Display formatting configuration
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

print("Environment libraries imported successfully.")
print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")


Environment libraries imported successfully.
pandas version: 3.0.6
numpy version:  2.5.3


---
## 3. Ingestion & Raw Dataset Preservation
We ensure the raw dataset is immutably preserved as `raw_data.csv` and load it into a pandas DataFrame.


In [2]:
# File path definitions
raw_data_path = Path("raw_data.csv")
source_data_path = Path("Unemployment in India.csv")

# Ensure raw_data.csv is preserved
if not raw_data_path.exists() and source_data_path.exists():
    shutil.copyfile(source_data_path, raw_data_path)
    print(" Created raw_data.csv from source.")
else:
    print(f"Verified raw dataset at: {raw_data_path.name}")

# Ingest raw data
df_raw = pd.read_csv(raw_data_path)
print(f"Raw dataset shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")


 Verified raw dataset at: raw_data.csv
Raw dataset shape: 768 rows x 7 columns


---
##  4. Dataset Structural Overview
We inspect the raw schema, data types, initial observations, and trailing records to diagnose structural anomalies.


In [3]:
print("=" * 60)
print("1. DATASET INFO & SCHEMA (df_raw.info())")
print("=" * 60)
df_raw.info()

print("\n" + "=" * 60)
print("2. FIRST 5 RECORDS (df_raw.head())")
print("=" * 60)
display(df_raw.head())

print("\n" + "=" * 60)
print("3. LAST 10 RECORDS (df_raw.tail(10)) — Note trailing empty lines")
print("=" * 60)
display(df_raw.tail(10))


1. DATASET INFO & SCHEMA (df_raw.info())
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Region                                    740 non-null    str    
 1    Date                                     740 non-null    str    
 2    Frequency                                740 non-null    str    
 3    Estimated Unemployment Rate (%)          740 non-null    float64
 4    Estimated Employed                       740 non-null    float64
 5    Estimated Labour Participation Rate (%)  740 non-null    float64
 6   Area                                      740 non-null    str    
dtypes: float64(3), str(4)
memory usage: 42.1 KB

2. FIRST 5 RECORDS (df_raw.head())


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural



3. LAST 10 RECORDS (df_raw.tail(10)) — Note trailing empty lines


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
758,NaN,NaN,NaN,NaN,NaN,NaN,NaN
759,NaN,NaN,NaN,NaN,NaN,NaN,NaN
760,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,NaN,NaN,NaN,NaN,NaN,NaN,NaN
762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
763,NaN,NaN,NaN,NaN,NaN,NaN,NaN
764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
765,NaN,NaN,NaN,NaN,NaN,NaN,NaN
766,NaN,NaN,NaN,NaN,NaN,NaN,NaN
767,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## 5. Data Dictionary

| Original Column Header | Cleaned Standard Header (`snake_case`) | Target Data Type | Nullable | Measurement Scale | Business Definition & Domain Interpretation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `Region` | `state` | `string` / `object` | No | Nominal Categorical | Name of the Indian State or Union Territory (28 unique administrative entities). |
| ` Date` | `date` | `datetime64[ns]` | No | Temporal (Date) | Observation month reference date (`YYYY-MM-DD`). Converted from raw string `DD-MM-YYYY`. |
| ` Frequency` | `frequency` | `string` / `object` | No | Nominal Categorical | Survey collection frequency (`Monthly`). Sanitized from whitespace anomalies. |
| ` Estimated Unemployment Rate (%)` | `unemployment_rate_pct` | `float64` | No | Continuous Ratio (%) | Proportion of the civilian labour force actively seeking employment during the reference month. |
| ` Estimated Employed` | `estimated_employed` | `int64` | No | Discrete Count | Estimated total headcount of individuals actively employed in the given state and area type. |
| ` Estimated Labour Participation Rate (%)` | `labour_participation_rate_pct` | `float64` | No | Continuous Ratio (%) | Proportion of the working-age population (15+ years) actively engaged in the labour force. |
| `Area` | `area_type` | `string` / `object` | No | Nominal Binary | Geographic stratification within each state: `Rural` or `Urban`. |


---
##  6. Missing-Value Audit & Remediation
We conduct a quantitative analysis of missing values across all columns.


In [4]:
# Quantitative Missingness Summary
null_counts = df_raw.isnull().sum()
null_percentages = (df_raw.isnull().sum() / len(df_raw)) * 100

missing_audit = pd.DataFrame({
    'Raw Column': df_raw.columns,
    'Null Count': null_counts.values,
    'Null Percentage (%)': null_percentages.values
})

print("=" * 60)
print("MISSING VALUES AUDIT TABLE:")
print("=" * 60)
display(missing_audit)

# Root-cause analysis of null records
all_null_rows = df_raw.isnull().all(axis=1).sum()
any_null_rows = df_raw.isnull().any(axis=1).sum()

print(f"\nTotal rows where ALL fields are null: {all_null_rows}")
print(f"Total rows where AT LEAST ONE field is null: {any_null_rows}")

# Engineering Decision:
# All 28 missing entries in each column originate from 28 completely blank trailing CSV artifact lines.
# We drop only completely empty rows using dropna(how='all') to retain all valid records without fabricating synthetic data.
df_clean = df_raw.copy()
initial_count = len(df_clean)
df_clean = df_clean.dropna(how='all')
print(f"Dropped {initial_count - len(df_clean)} empty trailing artifact rows. Retained {len(df_clean)} valid observations.")


MISSING VALUES AUDIT TABLE:


,Raw Column,Null Count,Null Percentage (%)
0,Region,28,3.645833
1,Date,28,3.645833
2,Frequency,28,3.645833
3,Estimated Unemployment Rate (%),28,3.645833
4,Estimated Employed,28,3.645833
5,Estimated Labour Participation Rate (%),28,3.645833
6,Area,28,3.645833



Total rows where ALL fields are null: 28
Total rows where AT LEAST ONE field is null: 28
Dropped 28 empty trailing artifact rows. Retained 740 valid observations.


---
## 7. Duplicate Records Audit & Resolution
We inspect the dataset for duplicated observational rows across the primary composite key (`state`, `area_type`, `date`).


In [5]:
# Check duplicates before dropping empty rows
raw_duplicates = df_raw.duplicated().sum()
print(f"Raw duplicates reported in initial df_raw: {raw_duplicates}")
print("-> Note: Raw duplicates (27) correspond to the 28 all-null trailing rows duplicating one another.")

# Verify duplicates in clean observational records
clean_duplicates = df_clean.duplicated().sum()
print(f"Duplicates among valid observation records: {clean_duplicates}")
assert clean_duplicates == 0, "Error: Unexpected duplicate records found!"
print("Verified 0 duplicate observational rows.")


Raw duplicates reported in initial df_raw: 27
-> Note: Raw duplicates (27) correspond to the 28 all-null trailing rows duplicating one another.
Duplicates among valid observation records: 0
 Verified 0 duplicate observational rows.


---
##  8. Header Renaming (`snake_case`) & Text Normalization
We sanitize column names to standard `snake_case` and strip leading/trailing whitespace across all text columns.


In [6]:
# 1. Clean and rename column headers to snake_case
df_clean.columns = df_clean.columns.str.strip()

column_mapping = {
    'Region': 'state',
    'Date': 'date',
    'Frequency': 'frequency',
    'Estimated Unemployment Rate (%)': 'unemployment_rate_pct',
    'Estimated Employed': 'estimated_employed',
    'Estimated Labour Participation Rate (%)': 'labour_participation_rate_pct',
    'Area': 'area_type'
}

df_clean = df_clean.rename(columns=column_mapping)
print(" Standardized Column Headers (snake_case):")
print(df_clean.columns.tolist())

# 2. Strip leading and trailing whitespace across all text columns
text_cols = ['state', 'frequency', 'area_type', 'date']
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 3. Verify category consolidation in 'frequency'
print(f"\nStandardized unique values in 'frequency': {df_clean['frequency'].unique().tolist()}")
print(f"Standardized unique values in 'area_type': {df_clean['area_type'].unique().tolist()}")
print(f"Unique state count: {df_clean['state'].nunique()}")


Standardized Column Headers (snake_case):
['state', 'date', 'frequency', 'unemployment_rate_pct', 'estimated_employed', 'labour_participation_rate_pct', 'area_type']

Standardized unique values in 'frequency': ['Monthly']
Standardized unique values in 'area_type': ['Rural', 'Urban']
Unique state count: 28


---
##  9. Datetime Conversion & Data Type Corrections
We parse the `date` strings into native `datetime64[ns]` with day-first format (`%d-%m-%Y`) and cast `estimated_employed` to integer headcount (`int64`).


In [7]:
# 1. Datetime conversion
df_clean['date'] = pd.to_datetime(df_clean['date'], format='%d-%m-%Y')
print("Converted 'date' column to datetime64[ns]")
print(f"   Date Range: {df_clean['date'].min().strftime('%Y-%m-%d')} to {df_clean['date'].max().strftime('%Y-%m-%d')}")

# 2. Cast estimated_employed to int64
df_clean['estimated_employed'] = df_clean['estimated_employed'].astype('int64')
print("Cast 'estimated_employed' headcount to int64")

# 3. Sort chronologically by date and state
df_clean = df_clean.sort_values(by=['date', 'state', 'area_type']).reset_index(drop=True)
print("Sorted dataset chronologically by ['date', 'state', 'area_type'].")

print("\n" + "=" * 60)
print("CLEANED SCHEMA (df_clean.dtypes):")
print("=" * 60)
print(df_clean.dtypes)


 Converted 'date' column to datetime64[ns]
   Date Range: 2019-05-31 to 2020-06-30
 Cast 'estimated_employed' headcount to int64
 Sorted dataset chronologically by ['date', 'state', 'area_type'].

CLEANED SCHEMA (df_clean.dtypes):
state                                       str
date                             datetime64[us]
frequency                                   str
unemployment_rate_pct                   float64
estimated_employed                        int64
labour_participation_rate_pct           float64
area_type                                   str
dtype: object


---
## 10. Descriptive Statistics & Integrity Assertions
We generate summary statistics for numerical and categorical variables, followed by programmatic quality assertions.


In [8]:
print("=" * 70)
print("1. NUMERICAL DESCRIPTIVE STATISTICS")
print("=" * 70)
display(df_clean.describe())

print("\n" + "=" * 70)
print("2. CATEGORICAL DESCRIPTIVE STATISTICS")
print("=" * 70)
display(df_clean.describe(include=['object', 'str']))

print("\n" + "=" * 70)
print("3. DATA INTEGRITY ASSERTION SUITE")
print("=" * 70)

# Assertion 1: Zero Nulls
assert df_clean.isnull().sum().sum() == 0, "Assertion Error: Null values remain!"

# Assertion 2: Value ranges
assert ((df_clean['unemployment_rate_pct'] >= 0) & (df_clean['unemployment_rate_pct'] <= 100)).all(), "Invalid unemployment rate!"
assert ((df_clean['labour_participation_rate_pct'] >= 0) & (df_clean['labour_participation_rate_pct'] <= 100)).all(), "Invalid labour participation!"
assert (df_clean['estimated_employed'] >= 0).all(), "Negative employed count detected!"

# Assertion 3: Composite uniqueness
composite_dups = df_clean.duplicated(subset=['state', 'area_type', 'date']).sum()
assert composite_dups == 0, "Duplicate composite primary key found!"

print("ALL 3/3 DATA INTEGRITY ASSERTIONS PASSED SUCCESSFULLY!")


1. NUMERICAL DESCRIPTIVE STATISTICS


,date,unemployment_rate_pct,estimated_employed,labour_participation_rate_pct
count,740,740.000000,7.400000e+02,740.000000
mean,2019-12-12 18:36:58.378378,11.787946,7.204460e+06,42.630122
min,2019-05-31 00:00:00,0.000000,4.942000e+04,13.330000
25%,2019-08-31 00:00:00,4.657500,1.190404e+06,38.062500
50%,2019-11-30 00:00:00,8.350000,4.744178e+06,41.160000
75%,2020-03-31 00:00:00,15.887500,1.127549e+07,45.505000
max,2020-06-30 00:00:00,76.740000,4.577751e+07,72.570000
std,NaN,10.721298,8.087988e+06,8.111094



2. CATEGORICAL DESCRIPTIVE STATISTICS


,state,frequency,area_type
count,740,740,740
unique,28,1,2
top,Andhra Pradesh,Monthly,Urban
freq,28,740,381



3. DATA INTEGRITY ASSERTION SUITE
 ALL 3/3 DATA INTEGRITY ASSERTIONS PASSED SUCCESSFULLY!


---
##  11. Serialization & Production Export
We export the final sanitized dataset to `cleaned_data.csv` without index and perform a round-trip reload verification.


In [9]:
output_csv_path = Path("cleaned_data.csv")
df_clean.to_csv(output_csv_path, index=False)
print(f"Successfully exported cleaned dataset to: {output_csv_path.name}")
print(f"   Exported file size: {output_csv_path.stat().st_size:,d} bytes")
print(f"   Exported dimensions: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

# Round-trip reload verification
verification_df = pd.read_csv(output_csv_path, parse_dates=['date'])
print(f"\nRound-trip reload verification:")
print(f"   Verified Shape: {verification_df.shape}")
print(f"   Verified Columns: {verification_df.columns.tolist()}")
print(f"   Verified Nulls: {verification_df.isnull().sum().sum()}")
display(verification_df.head(5))


 Successfully exported cleaned dataset to: cleaned_data.csv


   Exported file size: 41,104 bytes
   Exported dimensions: 740 rows x 7 columns



Round-trip reload verification:
   Verified Shape: (740, 7)
   Verified Columns: ['state', 'date', 'frequency', 'unemployment_rate_pct', 'estimated_employed', 'labour_participation_rate_pct', 'area_type']
   Verified Nulls: 0


,state,date,frequency,unemployment_rate_pct,estimated_employed,labour_participation_rate_pct,area_type
0,Andhra Pradesh,2019-05-31,Monthly,3.65,11999139,43.24,Rural
1,Andhra Pradesh,2019-05-31,Monthly,6.09,4788661,37.45,Urban
2,Assam,2019-05-31,Monthly,4.29,11749334,57.39,Rural
3,Assam,2019-05-31,Monthly,7.87,1671707,44.92,Urban
4,Bihar,2019-05-31,Monthly,9.27,24322330,39.75,Rural


---
## 🏁 12. Preprocessing Architecture Summary

| Metric | Raw Dataset (`raw_data.csv`) | Cleaned Dataset (`cleaned_data.csv`) | Transformation Applied |
| :--- | :--- | :--- | :--- |
| **Total Rows** | 768 | 740 | Dropped 28 empty trailing delimiter lines |
| **Total Columns** | 7 (with padding) | 7 (`snake_case`) | Cleaned headers & normalized names |
| **Missing Cells** | 196 (28 per col) | 0 | 100% complete observational records |
| **Duplicates** | 27 (empty rows) | 0 | Unique on `(state, area_type, date)` |
| **Date Column** | String (`DD-MM-YYYY`) | `datetime64[ns]` (`YYYY-MM-DD`) | ISO 8601 standardized format |
| **Employed Count** | Float (`.00`) | Integer (`int64`) | Exact discrete headcount representation |
| **Sorting** | Unsorted | Chronological | Sorted by `date`, `state`, `area_type` |

**Sign-off:** Data preparation and preprocessing are complete and ready for exploratory analysis and modeling.
